In [2]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image

# ======================
# 1) تابع محاسبه DICE
# ======================
def dice_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)      # تبدیل لاجیت خروجی به محدوده [0,1]
    pred = (pred > 0.5).float()       # باینری کردن خروجی
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.mean()

# ======================
# 2) تنظیم seed
# ======================
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ======================
# 3) تعریف دیتاست
# ======================
class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        img_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        # باینری کردن ماسک (فرض بر این است که مقادیر 0 و 255 هستند)
        mask = (mask > 0.5).float()

        return image, mask

# ======================
# 4) آدرس فولدر تصاویر و ماسک‌ها
# ======================
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))
assert len(images_list) == len(masks_list), "تعداد تصاویر با ماسک‌ها برابر نیست."

# ======================
# 5) تقسیم داده‌ها: 70% Train, 15% Val, 15% Test
# ======================
total_size = len(images_list)
train_size = int(total_size * 0.7)
val_size   = int(total_size * 0.15)
test_size  = total_size - train_size - val_size

train_images = images_list[:train_size]
train_masks  = masks_list[:train_size]

val_images = images_list[train_size:train_size + val_size]
val_masks  = masks_list[train_size:train_size + val_size]

test_images = images_list[train_size + val_size:]
test_masks  = masks_list[train_size + val_size:]

# ======================
# 6) تعریف ترنسفورم‌ها
# ======================
# ترنسفورم برای داده‌های تست و اعتبارسنجی
transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم پایه برای داده‌های آموزش
transform_train_base = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم آگومنت‌شده برای داده‌های آموزش
transform_train_aug = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor()
])

# ======================
# 7) ساخت دیتاست‌ها
# ======================
# برای آموزش، ترکیب دیتاست پایه و آگومنت‌شده
train_dataset_base = CorneaDataset(train_images, train_masks, transform=transform_train_base)
train_dataset_aug  = CorneaDataset(train_images, train_masks, transform=transform_train_aug)
train_dataset      = ConcatDataset([train_dataset_base, train_dataset_aug])

val_dataset   = CorneaDataset(val_images, val_masks, transform=transform_test)
test_dataset  = CorneaDataset(test_images, test_masks, transform=transform_test)

# ======================
# 8) ساخت DataLoader‌ها
# ======================
batch_size = 2
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 9) تعریف Attention Block برای Attention U-Net
# ======================
class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        """
        F_g: تعداد کانال‌های سیگنال gating (از decoder)
        F_l: تعداد کانال‌های سیگنال لایه skip (از encoder)
        F_int: تعداد کانال‌های میانی (معمولاً F_l // 2)
        """
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        # g: سیگنال gating از decoder
        # x: ویژگی‌های skip connection از encoder
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

# ======================
# 10) تعریف مدل Attention U-Net
# ======================
class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(AttentionUNet, self).__init__()
        # Encoder
        self.encoder1 = self.conv_block(in_channels, features[0])
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder2 = self.conv_block(features[0], features[1])
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder3 = self.conv_block(features[1], features[2])
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder4 = self.conv_block(features[2], features[3])
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = self.conv_block(features[3], features[3]*2)

        # Decoder با استفاده از Attention Gate
        self.up4 = nn.ConvTranspose2d(features[3]*2, features[3], kernel_size=2, stride=2)
        self.att4 = AttentionBlock(F_g=features[3], F_l=features[3], F_int=features[3]//2)
        self.decoder4 = self.conv_block(features[3]*2, features[3])

        self.up3 = nn.ConvTranspose2d(features[3], features[2], kernel_size=2, stride=2)
        self.att3 = AttentionBlock(F_g=features[2], F_l=features[2], F_int=features[2]//2)
        self.decoder3 = self.conv_block(features[2]*2, features[2])

        self.up2 = nn.ConvTranspose2d(features[2], features[1], kernel_size=2, stride=2)
        self.att2 = AttentionBlock(F_g=features[1], F_l=features[1], F_int=features[1]//2)
        self.decoder2 = self.conv_block(features[1]*2, features[1])

        self.up1 = nn.ConvTranspose2d(features[1], features[0], kernel_size=2, stride=2)
        self.att1 = AttentionBlock(F_g=features[0], F_l=features[0], F_int=features[0]//2)
        self.decoder1 = self.conv_block(features[0]*2, features[0])

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        """
        یک بلوک کانولوشنی استاندارد: Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU
        """
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)      # [B, 64, H, W]
        p1 = self.pool1(e1)        # [B, 64, H/2, W/2]

        e2 = self.encoder2(p1)     # [B, 128, H/2, W/2]
        p2 = self.pool2(e2)        # [B, 128, H/4, W/4]

        e3 = self.encoder3(p2)     # [B, 256, H/4, W/4]
        p3 = self.pool3(e3)        # [B, 256, H/8, W/8]

        e4 = self.encoder4(p3)     # [B, 512, H/8, W/8]
        p4 = self.pool4(e4)        # [B, 512, H/16, W/16]

        # Bottleneck
        b = self.bottleneck(p4)    # [B, 1024, H/16, W/16]

        # Decoder
        up4 = self.up4(b)          # [B, 512, H/8, W/8]
        att4 = self.att4(g=up4, x=e4)
        d4 = torch.cat([up4, att4], dim=1)  # [B, 1024, H/8, W/8]
        d4 = self.decoder4(d4)     # [B, 512, H/8, W/8]

        up3 = self.up3(d4)         # [B, 256, H/4, W/4]
        att3 = self.att3(g=up3, x=e3)
        d3 = torch.cat([up3, att3], dim=1)  # [B, 512, H/4, W/4]
        d3 = self.decoder3(d3)     # [B, 256, H/4, W/4]

        up2 = self.up2(d3)         # [B, 128, H/2, W/2]
        att2 = self.att2(g=up2, x=e2)
        d2 = torch.cat([up2, att2], dim=1)  # [B, 256, H/2, W/2]
        d2 = self.decoder2(d2)     # [B, 128, H/2, W/2]

        up1 = self.up1(d2)         # [B, 64, H, W]
        att1 = self.att1(g=up1, x=e1)
        d1 = torch.cat([up1, att1], dim=1)  # [B, 128, H, W]
        d1 = self.decoder1(d1)     # [B, 64, H, W]

        out = self.final_conv(d1)  # [B, out_channels, H, W]
        return out

# ======================
# 11) تنظیم دستگاه و مدل
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionUNet(in_channels=3, out_channels=1).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# 12) توابع آموزش و اعتبارسنجی
# ======================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        dice_score = dice_coefficient(outputs, masks)
        epoch_loss += loss.item()
        epoch_dice += dice_score.item()

    return epoch_loss / len(dataloader), epoch_dice / len(dataloader)

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    val_loss = 0.0
    val_dice = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            dice_score = dice_coefficient(outputs, masks)
            val_loss += loss.item()
            val_dice += dice_score.item()

    return val_loss / len(dataloader), val_dice / len(dataloader)

# ======================
# 13) حلقه آموزش و اعتبارسنجی
# ======================
num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ارزیابی نهایی روی Test Set
# ======================
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Dice: {test_dice:.4f}")

# ======================
# 15) ذخیره مدل
# ======================
torch.save(model.state_dict(), "attention_unet_cornea.pth")


In [3]:
batch_size = 4

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ارزیابی نهایی روی Test Set
# ======================
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Dice: {test_dice:.4f}")

# ======================
# 15) ذخیره مدل
# ======================
# torch.save(model.state_dict(), "attention_unet_cornea.pth")

In [1]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image

# ======================
# 1) تابع محاسبه DICE
# ======================
def dice_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)      # تبدیل لاجیت خروجی به محدوده [0,1]
    pred = (pred > 0.5).float()       # باینری کردن خروجی
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.mean()

# ======================
# 2) تنظیم seed
# ======================
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ======================
# 3) تعریف دیتاست
# ======================
class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        img_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        # باینری کردن ماسک (فرض بر این است که مقادیر 0 و 255 هستند)
        mask = (mask > 0.5).float()

        return image, mask

# ======================
# 4) آدرس فولدر تصاویر و ماسک‌ها
# ======================
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))
assert len(images_list) == len(masks_list), "تعداد تصاویر با ماسک‌ها برابر نیست."

# ======================
# 5) تقسیم داده‌ها: 70% Train, 15% Val, 15% Test
# ======================
total_size = len(images_list)
train_size = int(total_size * 0.7)
val_size   = int(total_size * 0.15)
test_size  = total_size - train_size - val_size

train_images = images_list[:train_size]
train_masks  = masks_list[:train_size]

val_images = images_list[train_size:train_size + val_size]
val_masks  = masks_list[train_size:train_size + val_size]

test_images = images_list[train_size + val_size:]
test_masks  = masks_list[train_size + val_size:]

# ======================
# 6) تعریف ترنسفورم‌ها
# ======================
# ترنسفورم برای داده‌های تست و اعتبارسنجی
transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم پایه برای داده‌های آموزش
transform_train_base = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم آگومنت‌شده برای داده‌های آموزش
transform_train_aug = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor()
])

# ======================
# 7) ساخت دیتاست‌ها
# ======================
# برای آموزش، ترکیب دیتاست پایه و آگومنت‌شده
train_dataset_base = CorneaDataset(train_images, train_masks, transform=transform_train_base)
train_dataset_aug  = CorneaDataset(train_images, train_masks, transform=transform_train_aug)
train_dataset      = ConcatDataset([train_dataset_base, train_dataset_aug])

val_dataset   = CorneaDataset(val_images, val_masks, transform=transform_test)
test_dataset  = CorneaDataset(test_images, test_masks, transform=transform_test)

# ======================
# 8) ساخت DataLoader‌ها
# ======================
batch_size = 2
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 9) تعریف Attention Block برای Attention U-Net
# ======================
class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        """
        F_g: تعداد کانال‌های سیگنال gating (از decoder)
        F_l: تعداد کانال‌های سیگنال لایه skip (از encoder)
        F_int: تعداد کانال‌های میانی (معمولاً F_l // 2)
        """
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        # g: سیگنال gating از decoder
        # x: ویژگی‌های skip connection از encoder
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

# ======================
# 10) تعریف مدل Attention U-Net
# ======================
class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(AttentionUNet, self).__init__()
        # Encoder
        self.encoder1 = self.conv_block(in_channels, features[0])
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder2 = self.conv_block(features[0], features[1])
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder3 = self.conv_block(features[1], features[2])
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder4 = self.conv_block(features[2], features[3])
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = self.conv_block(features[3], features[3]*2)

        # Decoder با استفاده از Attention Gate
        self.up4 = nn.ConvTranspose2d(features[3]*2, features[3], kernel_size=2, stride=2)
        self.att4 = AttentionBlock(F_g=features[3], F_l=features[3], F_int=features[3]//2)
        self.decoder4 = self.conv_block(features[3]*2, features[3])

        self.up3 = nn.ConvTranspose2d(features[3], features[2], kernel_size=2, stride=2)
        self.att3 = AttentionBlock(F_g=features[2], F_l=features[2], F_int=features[2]//2)
        self.decoder3 = self.conv_block(features[2]*2, features[2])

        self.up2 = nn.ConvTranspose2d(features[2], features[1], kernel_size=2, stride=2)
        self.att2 = AttentionBlock(F_g=features[1], F_l=features[1], F_int=features[1]//2)
        self.decoder2 = self.conv_block(features[1]*2, features[1])

        self.up1 = nn.ConvTranspose2d(features[1], features[0], kernel_size=2, stride=2)
        self.att1 = AttentionBlock(F_g=features[0], F_l=features[0], F_int=features[0]//2)
        self.decoder1 = self.conv_block(features[0]*2, features[0])

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        """
        یک بلوک کانولوشنی استاندارد: Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU
        """
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)      # [B, 64, H, W]
        p1 = self.pool1(e1)        # [B, 64, H/2, W/2]

        e2 = self.encoder2(p1)     # [B, 128, H/2, W/2]
        p2 = self.pool2(e2)        # [B, 128, H/4, W/4]

        e3 = self.encoder3(p2)     # [B, 256, H/4, W/4]
        p3 = self.pool3(e3)        # [B, 256, H/8, W/8]

        e4 = self.encoder4(p3)     # [B, 512, H/8, W/8]
        p4 = self.pool4(e4)        # [B, 512, H/16, W/16]

        # Bottleneck
        b = self.bottleneck(p4)    # [B, 1024, H/16, W/16]

        # Decoder
        up4 = self.up4(b)          # [B, 512, H/8, W/8]
        att4 = self.att4(g=up4, x=e4)
        d4 = torch.cat([up4, att4], dim=1)  # [B, 1024, H/8, W/8]
        d4 = self.decoder4(d4)     # [B, 512, H/8, W/8]

        up3 = self.up3(d4)         # [B, 256, H/4, W/4]
        att3 = self.att3(g=up3, x=e3)
        d3 = torch.cat([up3, att3], dim=1)  # [B, 512, H/4, W/4]
        d3 = self.decoder3(d3)     # [B, 256, H/4, W/4]

        up2 = self.up2(d3)         # [B, 128, H/2, W/2]
        att2 = self.att2(g=up2, x=e2)
        d2 = torch.cat([up2, att2], dim=1)  # [B, 256, H/2, W/2]
        d2 = self.decoder2(d2)     # [B, 128, H/2, W/2]

        up1 = self.up1(d2)         # [B, 64, H, W]
        att1 = self.att1(g=up1, x=e1)
        d1 = torch.cat([up1, att1], dim=1)  # [B, 128, H, W]
        d1 = self.decoder1(d1)     # [B, 64, H, W]

        out = self.final_conv(d1)  # [B, out_channels, H, W]
        return out

# ======================
# 11) تنظیم دستگاه و مدل
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionUNet(in_channels=3, out_channels=1).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# 12) توابع آموزش و اعتبارسنجی
# ======================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        dice_score = dice_coefficient(outputs, masks)
        epoch_loss += loss.item()
        epoch_dice += dice_score.item()

    return epoch_loss / len(dataloader), epoch_dice / len(dataloader)

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    val_loss = 0.0
    val_dice = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            dice_score = dice_coefficient(outputs, masks)
            val_loss += loss.item()
            val_dice += dice_score.item()

    return val_loss / len(dataloader), val_dice / len(dataloader)

# ======================
# 13) حلقه آموزش و اعتبارسنجی
# ======================
num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ارزیابی نهایی روی Test Set
# ======================
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Dice: {test_dice:.4f}")

# ======================
# 15) ذخیره مدل
# ======================
# torch.save(model.state_dict(), "attention_unet_cornea.pth")



batch_size = 8

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ارزیابی نهایی روی Test Set
# ======================
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Dice: {test_dice:.4f}")

# ======================
# 15) ذخیره مدل
# ======================
# torch.save(model.state_dict(), "attention_unet_cornea.pth")

In [1]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image

# ======================
# 1) تابع محاسبه DICE
# ======================
def dice_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)      # تبدیل لاجیت خروجی به محدوده [0,1]
    pred = (pred > 0.5).float()       # باینری کردن خروجی
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.mean()

# ======================
# 2) تنظیم seed
# ======================
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ======================
# 3) تعریف دیتاست
# ======================
class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        img_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        # باینری کردن ماسک (فرض بر این است که مقادیر 0 و 255 هستند)
        mask = (mask > 0.5).float()

        return image, mask

# ======================
# 4) آدرس فولدر تصاویر و ماسک‌ها
# ======================
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))
assert len(images_list) == len(masks_list), "تعداد تصاویر با ماسک‌ها برابر نیست."

# ======================
# 5) تقسیم داده‌ها: 70% Train, 15% Val, 15% Test
# ======================
total_size = len(images_list)
train_size = int(total_size * 0.7)
val_size   = int(total_size * 0.15)
test_size  = total_size - train_size - val_size

train_images = images_list[:train_size]
train_masks  = masks_list[:train_size]

val_images = images_list[train_size:train_size + val_size]
val_masks  = masks_list[train_size:train_size + val_size]

test_images = images_list[train_size + val_size:]
test_masks  = masks_list[train_size + val_size:]

# ======================
# 6) تعریف ترنسفورم‌ها
# ======================
# ترنسفورم برای داده‌های تست و اعتبارسنجی
transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم پایه برای داده‌های آموزش
transform_train_base = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم آگومنت‌شده برای داده‌های آموزش
transform_train_aug = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor()
])

# ======================
# 7) ساخت دیتاست‌ها
# ======================
# برای آموزش، ترکیب دیتاست پایه و آگومنت‌شده
train_dataset_base = CorneaDataset(train_images, train_masks, transform=transform_train_base)
train_dataset_aug  = CorneaDataset(train_images, train_masks, transform=transform_train_aug)
train_dataset      = ConcatDataset([train_dataset_base, train_dataset_aug])

val_dataset   = CorneaDataset(val_images, val_masks, transform=transform_test)
test_dataset  = CorneaDataset(test_images, test_masks, transform=transform_test)

# ======================
# 8) ساخت DataLoader‌ها
# ======================
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 9) تعریف Attention Block برای Attention U-Net
# ======================
class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        """
        F_g: تعداد کانال‌های سیگنال gating (از decoder)
        F_l: تعداد کانال‌های سیگنال لایه skip (از encoder)
        F_int: تعداد کانال‌های میانی (معمولاً F_l // 2)
        """
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        # g: سیگنال gating از decoder
        # x: ویژگی‌های skip connection از encoder
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

# ======================
# 10) تعریف مدل Attention U-Net
# ======================
class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(AttentionUNet, self).__init__()
        # Encoder
        self.encoder1 = self.conv_block(in_channels, features[0])
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder2 = self.conv_block(features[0], features[1])
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder3 = self.conv_block(features[1], features[2])
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder4 = self.conv_block(features[2], features[3])
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = self.conv_block(features[3], features[3]*2)

        # Decoder با استفاده از Attention Gate
        self.up4 = nn.ConvTranspose2d(features[3]*2, features[3], kernel_size=2, stride=2)
        self.att4 = AttentionBlock(F_g=features[3], F_l=features[3], F_int=features[3]//2)
        self.decoder4 = self.conv_block(features[3]*2, features[3])

        self.up3 = nn.ConvTranspose2d(features[3], features[2], kernel_size=2, stride=2)
        self.att3 = AttentionBlock(F_g=features[2], F_l=features[2], F_int=features[2]//2)
        self.decoder3 = self.conv_block(features[2]*2, features[2])

        self.up2 = nn.ConvTranspose2d(features[2], features[1], kernel_size=2, stride=2)
        self.att2 = AttentionBlock(F_g=features[1], F_l=features[1], F_int=features[1]//2)
        self.decoder2 = self.conv_block(features[1]*2, features[1])

        self.up1 = nn.ConvTranspose2d(features[1], features[0], kernel_size=2, stride=2)
        self.att1 = AttentionBlock(F_g=features[0], F_l=features[0], F_int=features[0]//2)
        self.decoder1 = self.conv_block(features[0]*2, features[0])

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        """
        یک بلوک کانولوشنی استاندارد: Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU
        """
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)      # [B, 64, H, W]
        p1 = self.pool1(e1)        # [B, 64, H/2, W/2]

        e2 = self.encoder2(p1)     # [B, 128, H/2, W/2]
        p2 = self.pool2(e2)        # [B, 128, H/4, W/4]

        e3 = self.encoder3(p2)     # [B, 256, H/4, W/4]
        p3 = self.pool3(e3)        # [B, 256, H/8, W/8]

        e4 = self.encoder4(p3)     # [B, 512, H/8, W/8]
        p4 = self.pool4(e4)        # [B, 512, H/16, W/16]

        # Bottleneck
        b = self.bottleneck(p4)    # [B, 1024, H/16, W/16]

        # Decoder
        up4 = self.up4(b)          # [B, 512, H/8, W/8]
        att4 = self.att4(g=up4, x=e4)
        d4 = torch.cat([up4, att4], dim=1)  # [B, 1024, H/8, W/8]
        d4 = self.decoder4(d4)     # [B, 512, H/8, W/8]

        up3 = self.up3(d4)         # [B, 256, H/4, W/4]
        att3 = self.att3(g=up3, x=e3)
        d3 = torch.cat([up3, att3], dim=1)  # [B, 512, H/4, W/4]
        d3 = self.decoder3(d3)     # [B, 256, H/4, W/4]

        up2 = self.up2(d3)         # [B, 128, H/2, W/2]
        att2 = self.att2(g=up2, x=e2)
        d2 = torch.cat([up2, att2], dim=1)  # [B, 256, H/2, W/2]
        d2 = self.decoder2(d2)     # [B, 128, H/2, W/2]

        up1 = self.up1(d2)         # [B, 64, H, W]
        att1 = self.att1(g=up1, x=e1)
        d1 = torch.cat([up1, att1], dim=1)  # [B, 128, H, W]
        d1 = self.decoder1(d1)     # [B, 64, H, W]

        out = self.final_conv(d1)  # [B, out_channels, H, W]
        return out

# ======================
# 11) تنظیم دستگاه و مدل
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionUNet(in_channels=3, out_channels=1).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# 12) توابع آموزش و اعتبارسنجی
# ======================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        dice_score = dice_coefficient(outputs, masks)
        epoch_loss += loss.item()
        epoch_dice += dice_score.item()

    return epoch_loss / len(dataloader), epoch_dice / len(dataloader)

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    val_loss = 0.0
    val_dice = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            dice_score = dice_coefficient(outputs, masks)
            val_loss += loss.item()
            val_dice += dice_score.item()

    return val_loss / len(dataloader), val_dice / len(dataloader)

# ======================
# 13) حلقه آموزش و اعتبارسنجی
# ======================
num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ارزیابی نهایی روی Test Set
# ======================
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Dice: {test_dice:.4f}")

# ======================
# 15) ذخیره مدل
# ======================
# torch.save(model.state_dict(), "attention_unet_cornea.pth")



batch_size = 8

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ارزیابی نهایی روی Test Set
# ======================
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Dice: {test_dice:.4f}")

# ======================
# 15) ذخیره مدل
# ======================
# torch.save(model.state_dict(), "attention_unet_cornea.pth")

In [2]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ارزیابی نهایی روی Test Set
# ======================
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Dice: {test_dice:.4f}")

# ======================
# 15) ذخیره مدل
# ======================
# torch.save(model.state_dict(), "attention_unet_cornea.pth")